In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "data" / "reference").is_dir():
    candidate = EXPERIMENT_DIR.parent
    if (candidate / "data" / "reference").is_dir():
        EXPERIMENT_DIR = candidate
if not (EXPERIMENT_DIR / "data" / "reference").is_dir():
    raise RuntimeError("Start Jupyter in this experiment directory or its notebooks directory.")

REFERENCE_DATA_DIR = EXPERIMENT_DIR / "data" / "reference"
GENERATED_DATA_DIR = EXPERIMENT_DIR / "runs" / "generated-data"
REFERENCE_MODEL_DIR = EXPERIMENT_DIR / "models" / "reference"
TRAINED_MODEL_DIR = EXPERIMENT_DIR / "runs" / "trained-models"
PROJECT_DIR = EXPERIMENT_DIR.parents[3]
EXPERIMENT_ID = EXPERIMENT_DIR.relative_to(PROJECT_DIR / "experiments")
RESULT_DIR = PROJECT_DIR / "figures" / "paper"
for directory in (GENERATED_DATA_DIR, TRAINED_MODEL_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# True reproduces the paper. Set False to regenerate data and retrain.
REPRODUCE_PAPER = True

def data_path(filename):
    root = REFERENCE_DATA_DIR if REPRODUCE_PAPER else GENERATED_DATA_DIR
    return root / filename

def model_path(filename):
    root = REFERENCE_MODEL_DIR if REPRODUCE_PAPER else TRAINED_MODEL_DIR
    return root / filename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import TensorDataset, DataLoader

### Generate data

In [ ]:
# Define variables
k = 1            # spring constant
m = 1            # mass 
b = 0.02        # damping coefficient
noise_var = 0.04
omega_0 = np.sqrt(k / m)
beta = b / (2 * m)
w1 = np.sqrt(omega_0**2 - beta**2)  # Damped natural frequency

In [ ]:
def init_conditions(num_samples, energy_range=(0.2,1)):
    energies = np.random.uniform(energy_range[0], energy_range[1], num_samples)
    phis = np.random.uniform(0, 2 * np.pi, num_samples)
    As = []
    
    for E in energies:
        Amp = np.sqrt(2 * E / k)
        As.append(Amp)

    return np.array(As), phis

def position(A, phi, t):
    return A * np.exp(-beta * t) * np.cos(w1 * t + phi)

def momentum(A, phi, t):
    return -A * m * np.exp(-beta * t) * (beta * np.cos(w1 * t + phi) + w1 * np.sin(w1 * t + phi))

def get_derivative(x_curr, x_prev, time_step):
    return (x_curr - x_prev)/time_step

def compute_derivatives(values : np.array, time_step):
    return np.gradient(values, time_step)

def add_noise(qs, ps):
    return qs + np.random.normal(0, noise_var, len(qs)), ps + np.random.normal(0, noise_var, len(ps))

def generate_damped_oscillator_data(num_trajectories, time_step, noisy=False, num_data=314, sample_all=False, sample_every=100, energy_range=(0.2,1)):
    As_0, phis_0 = init_conditions(num_trajectories, energy_range)
    
    qs, ps, ts = [], [], []
    q_dots, p_dots = [], []
    
    for A0, phi_0 in zip(As_0, phis_0):
        t_vals = np.arange(0, num_data, time_step)

        if noisy:
            qs_traj, ps_traj = add_noise(position(A0, phi_0, t_vals), momentum(A0, phi_0, t_vals))
        else:
            qs_traj, ps_traj = position(A0, phi_0, t_vals), momentum(A0, phi_0, t_vals)
        
        q_dot, p_dot = compute_derivatives(qs_traj, time_step), compute_derivatives(ps_traj, time_step)

        if sample_all: 
            q_dots.append(q_dot)
            p_dots.append(p_dot)
            qs.append(qs_traj)
            ps.append(ps_traj)
            ts.append(t_vals)
        else:        
            # Sample every 100th data 
            q_dots.append(q_dot[::sample_every])
            p_dots.append(p_dot[::sample_every])
            qs.append(qs_traj[::sample_every])
            ps.append(ps_traj[::sample_every])
            ts.append(t_vals[::sample_every])
    
    return np.asarray(qs), np.asarray(ps), np.asarray(q_dots), np.asarray(p_dots), np.asarray(ts)

#### Generate Training Data

In [ ]:
# Parameters
num_trajectories = 30                       # 40 sets
num_data = 314                              # Each with 100 time steps
time_step = 0.1  

# Generate data
qs, ps, q_dots, p_dots, ts = generate_damped_oscillator_data(num_trajectories, time_step, noisy=True, sample_every=10)

# Reshape into single dataset
qs_flat = qs.reshape(-1)
ps_flat = ps.reshape(-1)
ts_flat = ts.reshape(-1)
q_dots_flat = q_dots.reshape(-1)
p_dots_flat = p_dots.reshape(-1)

# Create DataFrame
train_df = pd.DataFrame({
    "q": qs_flat,
    "p": ps_flat,
    "q_dot": q_dots_flat,
    "p_dot": p_dots_flat
})

train_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_train_noise.csv", index=False)

#### Generate Test Data

In [ ]:
qs_test, ps_test, q_dots_test, p_dots_test, ts_test = generate_damped_oscillator_data(1, time_step, noisy=False, sample_all=True, energy_range=(0.2,0.2))
qs_test_flat = qs_test.reshape(-1)
ps_test_flat = ps_test.reshape(-1)
ts_test_flat = ts_test.reshape(-1)
q_dots_test_flat = q_dots_test.reshape(-1)
p_dots_test_flat = p_dots_test.reshape(-1)

test_df = pd.DataFrame({"q": qs_test_flat, "p": ps_test_flat, "q_dot": q_dots_test_flat, "p_dot": p_dots_test_flat})
test_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_low_energy.csv", index=False)

qs_test, ps_test, q_dots_test, p_dots_test, ts_test = generate_damped_oscillator_data(1, time_step, sample_all=True, energy_range=(1.0, 1.0))
qs_test_flat = qs_test.reshape(-1)
ps_test_flat = ps_test.reshape(-1)
ts_test_flat = ts_test.reshape(-1)
q_dots_test_flat = q_dots_test.reshape(-1)
p_dots_test_flat = p_dots_test.reshape(-1)

test_df = pd.DataFrame({"q": qs_test_flat, "p": ps_test_flat, "q_dot": q_dots_test_flat, "p_dot": p_dots_test_flat})
test_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_high_energy.csv", index=False)

### Load Data

In [ ]:
train_df = pd.read_csv(data_path("damped_oscillator_train_noise.csv"))
test_df_low = pd.read_csv(data_path("damped_oscillator_low_energy.csv"))
test_df_high = pd.read_csv(data_path("damped_oscillator_high_energy.csv"))

### Prepare training data 

In [ ]:
features = train_df.iloc[:, 0:2]
targets = train_df.iloc[:, 2:]

In [ ]:
scaler = MinMaxScaler(feature_range=(-1,1))
#features = scaler.fit_transform(features)

In [ ]:
f_tensor = torch.FloatTensor(features.values)
t_tensor = torch.FloatTensor(targets.values)

In [ ]:
train_dataset = TensorDataset(f_tensor, t_tensor)
train_dl = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)

### Constants

In [ ]:
HIDDEN_DIM = 200
LEARNING_RATE = 1e-3
EPOCHS = 500
if REPRODUCE_PAPER:
    EPOCHS = 0
INPUT_DIM = 2
OUTPUT_DIM = 1 
OUTPUT_DIM_B = 2
HIDDEN_LAYERS_DIM = 5

### Model Definitions

In [ ]:
# Build Neural Network
class BaselineNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM_B, hidden_layers=HIDDEN_LAYERS_DIM):
        super(BaselineNN, self).__init__()
        
        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh

    def forward(self, x):
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x
        
class HNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, hidden_layers=HIDDEN_LAYERS_DIM):
        super(HNN, self).__init__()

        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh

    def forward(self, x):
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x

    def grad_forward(self, x):
        # Ensure that x requires gradient.
        x.requires_grad_(True)
        H = self.forward(x)
        grad_outputs = torch.ones_like(H)
        
        grad_H = torch.autograd.grad(outputs=H, inputs=x, grad_outputs=grad_outputs, create_graph=True)[0]
        dhdq = grad_H[:,0]
        dhdp = grad_H[:,1]

        dpdt = -dhdq
        dqdt = dhdp
        
        return dpdt, dqdt

In [ ]:
def train_baselineNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state_baseline = None
    min_loss = float('inf')
    model = BaselineNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM)
    
    # Define the mean squared error loss.
    # criterion = nn.MSELoss()
    criterion = nn.SmoothL1Loss()
    
    # Create an Adam optimizer.
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    model.train()
    
    for epoch in range(epochs):
        for inputs, targets in data_dl:
            optimizer.zero_grad()
            outputs = model(inputs) 
            loss = criterion(outputs, targets)
            loss.backward()  
            optimizer.step()

        if epoch % 100 == 0:
            print(f"Epoch: {epoch}/{epochs}, Loss: {loss.item()}") 

        if loss.item() < min_loss: 
            min_loss = loss.item()
            best_model_state_baseline = copy.deepcopy(model.state_dict())

    return model, best_model_state_baseline
    
def train_HNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state = None
    min_loss = float('inf')
    model = HNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM)
    
    # Define the mean squared error loss.
    # criterion = nn.MSELoss()
    criterion = nn.SmoothL1Loss()
    
    # Create an Adam optimizer.
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    model.train()
    
    for epoch in range(epochs):
        for inputs, targets in data_dl:
            optimizer.zero_grad()
            
            dpdt, dqdt = model.grad_forward(inputs)
            predictions = torch.stack([dqdt, dpdt], dim=1) 
            
            loss = criterion(predictions, targets)
            loss.backward()
            
            optimizer.step()

        if epoch % 100 == 0:
            print(f"Epoch: {epoch}/{epochs}, Loss: {loss.item()}") 

        if loss.item() < min_loss: 
            min_loss = loss.item()
            best_model_state = copy.deepcopy(model.state_dict())

    return model, best_model_state

### Train model

In [ ]:
_, best_model_state_baseline = train_baselineNN(train_dl)

In [ ]:
_, best_model_state = train_HNN(train_dl)

In [ ]:
# Save the best model state for baseline if it was updated
if best_model_state_baseline is not None: 
    torch.save(best_model_state_baseline, TRAINED_MODEL_DIR / "best_model_baseline_noise.pth")

# Load the best model 
best_model_baseline = BaselineNN()
best_model_baseline.load_state_dict(torch.load(model_path("best_model_baseline_noise.pth"), map_location="cpu"), strict=False)

In [ ]:
# Save the best model state if it was updated
if best_model_state is not None: 
    torch.save(best_model_state, TRAINED_MODEL_DIR / "best_model_hnn_noise.pth")

In [ ]:
# Load the best model 
best_model_hnn = HNN()
best_model_hnn.load_state_dict(torch.load(model_path("best_model_hnn_noise.pth"), map_location="cpu"), strict=False)

### Prepare testing data 

In [ ]:
features_test = test_df_high.iloc[:, 0:2]
targets_test = test_df_high.iloc[:, 2:]

#features_test_scaled = scaler.fit_transform(features_test)
f_tensor_test = torch.FloatTensor(features_test.values)
t_tensor_test = torch.FloatTensor(targets_test.values)

test_dataset = TensorDataset(f_tensor_test, t_tensor_test)
test_dl_high = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [ ]:
features_test_l = test_df_low.iloc[:, 0:2]
targets_test_l = test_df_low.iloc[:, 2:]

#features_test_scaled = scaler.fit_transform(features_test)
f_tensor_test_l = torch.FloatTensor(features_test_l.values)
t_tensor_test_l = torch.FloatTensor(targets_test_l.values)

test_dataset_l = TensorDataset(f_tensor_test_l, t_tensor_test_l)
test_dl_low = DataLoader(test_dataset_l, batch_size=len(test_dataset_l), shuffle=False)

### Plot results

In [ ]:
def evaluate_model_baseline(model, data_dl):
    model.eval()
    predictions = []
    true = []

    for test_inps, test_targets in data_dl: 
        outputs = model(test_inps)
        pred_batch = outputs.detach().numpy()
        true_batch = test_targets.detach().numpy()

        predictions.append(pred_batch)
        true.append(true_batch)

    y_true = np.concatenate(true, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  

    return y_pred, y_true

def evaluate_model(model, data_dl):
    model.eval()
    predictions = []
    true = []
    Hs = []

    for test_inps, test_targets in data_dl: 
        H = model(test_inps)
        dpdt, dqdt = model.grad_forward(test_inps)
        pred_batch = torch.stack([dqdt, dpdt], dim=1).detach().numpy()
        true_batch = test_targets.detach().numpy()
        H = H.detach().numpy()

        predictions.append(pred_batch)
        true.append(true_batch)
        Hs.append(H)

    y_true = np.concatenate(true, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  
    H = np.concatenate(H, axis=0)

    return y_pred, y_true, H

In [ ]:
y_pred_base_high, y_true_base_high = evaluate_model_baseline(best_model_baseline, test_dl_high)
y_pred_base_low, y_true_base_low = evaluate_model_baseline(best_model_baseline, test_dl_low)

y_pred_hnn_high, y_true_hnn_high, H_high = evaluate_model(best_model_hnn, test_dl_high)
y_pred_hnn_low, y_true_hnn_low, H_low = evaluate_model(best_model_hnn, test_dl_low)

In [ ]:
# plt.plot(np.arange(len(y_pred_hnn_low)), H_low);

In [ ]:
mse_baseline_high = (y_pred_base_high[:,0] - y_true_base_high[:,0])**2 + (y_pred_base_high[:,1] - y_true_base_high[:,1])**2
mse_baseline_low = (y_pred_base_low[:,0] - y_true_base_low[:,0])**2 + (y_pred_base_low[:,1] - y_true_base_low[:,1])**2

mse_hnn_high = (y_pred_hnn_high[:,0] - y_true_hnn_high[:,0])**2 + (y_pred_hnn_high[:,1] - y_true_hnn_high[:,1])**2
mse_hnn_low = (y_pred_hnn_low[:,0] - y_true_hnn_low[:,0])**2 + (y_pred_hnn_low[:,1] - y_true_hnn_low[:,1])**2

In [ ]:
n_points = 30
x_high = np.arange(n_points)
x_low = np.arange(n_points)

cumsum_baseline_high = mse_baseline_high[:n_points].cumsum()
cumsum_hnn_high = mse_hnn_high[:n_points].cumsum()
cumsum_baseline_low = mse_baseline_low[:n_points].cumsum()
cumsum_hnn_low = mse_hnn_low[:n_points].cumsum()

# Create two subplots: one for high and one for low comparisons
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# ax1.plot(x_high, cumsum_baseline_high, label="Baseline High Energy", color="red")
# ax1.plot(x_high, cumsum_hnn_high, label="HNN High Energy", color="blue")
# ax1.set_title("High Energy MSE Comparison")
# ax1.set_ylabel("Cumulative Sum of MSE")
# ax1.legend()

# ax2.plot(x_low, cumsum_baseline_low, label="Baseline Low Energy", color="red")
# ax2.plot(x_low, cumsum_hnn_low, label="HNN Low Energy", color="blue")
# ax2.set_title("Low Energy MSE Comparison")
# ax2.set_ylabel("Cumulative Sum of MSE")
# ax2.legend()

# plt.tight_layout()
# plt.show()

In [ ]:
def plot_phase_space_dot(y_true, y_pred, save_path=None):
    q_dot_true = y_true[:, 0]
    p_dot_true = y_true[:, 1]
    q_dot_pred = y_pred[:, 0]
    p_dot_pred = y_pred[:, 1]

    plt.figure(figsize=(7, 5))
    plt.plot(q_dot_true, p_dot_true, label='True', color='red')
    plt.plot(q_dot_pred, p_dot_pred, alpha=0.6, label='Predicted', color='blue', linestyle='--')
    plt.xlabel(r'$\dot{q}$', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.ylabel(r'$\dot{p}$', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12, fontname='Times New Roman')
    plt.yticks(fontsize=12, fontname='Times New Roman')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

def plot_true_vs_pred_ax(ax, y_true, y_pred, color, title):
    q_dot_true = y_true[:, 0]
    p_dot_true = y_true[:, 1]
    q_dot_pred = y_pred[:, 0]
    p_dot_pred = y_pred[:, 1]
    
    ax.plot(q_dot_true, p_dot_true, alpha=0.3, label='True', color="black")
    ax.plot(q_dot_pred, p_dot_pred, alpha=1.0, label='Predicted', color=color)
    ax.set_xlabel('q_dot')
    ax.set_ylabel('p_dot')
    ax.set_title(title)
    ax.legend()

In [ ]:
# plot_phase_space_dot(y_true_base_low[:1000], y_pred_base_low[:1000])

In [ ]:
plot_phase_space_dot(y_true_hnn_low[:1000], y_pred_hnn_low[:1000], save_path=RESULT_DIR / "f6d.png")

In [ ]:
# plot_phase_space_dot(y_true_base_high[:1000], y_pred_base_high[:1000])

In [ ]:
# plot_phase_space_dot(y_true_hnn_high[:1000], y_pred_hnn_high[:1000])

In [ ]:
def plot_qdot_time(y_true, y_pred, plot_true=True, plot_pred=True, n=250, save_path=None):
    q_dot_true = y_true[:, 0]
    q_dot_pred = y_pred[:, 0]
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(q_dot_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(q_dot_pred[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.ylabel(r'$\dot{q}$', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12, fontname='Times New Roman')
    plt.yticks(fontsize=12, fontname='Times New Roman')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# baseline low energy
# plot_qdot_time(y_true_base_low, y_pred_base_low, n=750)

In [ ]:
# hnn low energy
plot_qdot_time(y_true_hnn_low, y_pred_hnn_low, n=750, save_path=RESULT_DIR / "f6b.png")

In [ ]:
# baseline high energy
# plot_qdot_time(y_true_base_high, y_pred_base_high, n=750)

In [ ]:
# hnn high energy
# plot_qdot_time(y_true_hnn_high, y_pred_hnn_high, n=750)

In [ ]:
def plot_hamiltonian_high(test_df, H_pred, plot_true=True, plot_pred=True, n=250, save_path=None):
    q = test_df["q"].values
    p = test_df["p"].values
    t = np.arange(0, num_data, time_step)
    H_true = (p**2 / (2 * m) + 0.5 * k * q**2) * np.exp(-2 * beta * t)
    H_pred = H_pred + 6.75
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(H_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(H_pred[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.ylabel('Hamiltonian', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12, fontname='Times New Roman')
    plt.yticks(fontsize=12, fontname='Times New Roman')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# plot_hamiltonian_high(test_df_high, H_high, n = len(H_high))

In [ ]:
def plot_hamiltonian_low(test_df, H_pred, plot_true=True, plot_pred=True, n=250, save_path=None):
    q = test_df["q"].values
    p = test_df["p"].values
    t = np.arange(0, num_data, time_step)
    H_true = (p**2 / (2 * m) + 0.5 * k * q**2) * np.exp(-2 * beta * t)
    H_pred = H_pred + 6.755
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(H_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(H_pred[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.ylabel('Hamiltonian', fontsize=14, fontweight='bold', fontname='Times New Roman')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12, fontname='Times New Roman')
    plt.yticks(fontsize=12, fontname='Times New Roman')
    plt.grid(True, linestyle='--', alpha=0.6)

    # plt.ylim((-0.3, 0.3))
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
plot_hamiltonian_low(test_df_low, H_low, n = len(H_low), save_path=RESULT_DIR / "f6f.png")